# Week 7 · Your Week 6 work, reviewed, and how this week works

I read your Week 6 log (Nov 24–27, around a wisdom-tooth extraction), every cell you added to the notebooks, and your note about the pictures. This notebook is short: §A goes through your notes, §B is what the meeting with Prof. Yoon changes, §C is this week's coding, and §D is how the pictures are drawn from now on. Do the warm-up (max pooling, which you said was the hard part of `04`) and start `01`.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "00")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** This week. `01` reuses a trained network on a new dataset; `02` learns a compact description of images without labels; `03` turns it into a model that can generate new images; `04` uses all of it to flag faulty parts. `01` §2 (domain shift) and `04` (anomaly detection) are the two topics Prof. Yoon's lab publishes on most.

In [ ]:
fig_week7_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 18.5 (max pooling) from memory | the part of `04` you found hard, spaced two days later |
| A | your Week 6 notes, checked | what you wrote, what's right, and one small correction |
| B | after the meeting | the exam is scored now, so the plan aims at a high score |
| C | this week's coding | which booklet blocks and which new blocks |
| D | pictures from now on | "put more effort into the figures": what changes, with your receptive-field drawing as the standard |

## 0. Recap first · 10 min, closed book

**Quick check:** Inverted dropout with keep probability $q = 0.8$: a unit outputs $a = 5$ and is kept. What does it pass on during training, and at test time?

<details><summary>Answer</summary>

Training: $5 / 0.8 = 6.25$. Test: $5$, unchanged. Dividing by $q$ during training makes the expected value $q \cdot a/q = a$, which is what the test-time network outputs. (Your own explanation in `02`.) [Week 6 · 02 §3 ↗](../week6/02_regularization.ipynb)

</details>

**Quick check:** Why can't batch norm use the batch's own mean and variance at test time? Give both problems.

<details><summary>Answer</summary>

(A) A batch of one image: the mean equals the value, so every feature becomes 0. (B) The same image would get a different output depending on which other images share its batch. Fix: the running averages collected during training. (Your Problems A and B in `03`.) [Week 6 · 03 §2 ↗](../week6/03_training_stability.ipynb)

</details>

**Quick check:** `Conv2d(3, 16, 5)` on `(N, 3, 32, 32)`, no padding: output shape and number of parameters?

<details><summary>Answer</summary>

$32 - 5 + 1 = 28$: `(N, 16, 28, 28)`. Parameters $16 \cdot 3 \cdot 25 + 16 = 1216$. [Week 6 · 04 §4 ↗](../week6/04_convolution.ipynb)

</details>

**Quick check:** Three stacked 3×3 convolutions (stride 1): how big is the patch of the image that one output value sees?

<details><summary>Answer</summary>

$7 \times 7$: each layer adds one pixel on each side, $(2n + 1) \times (2n + 1)$ with $n = 3$. (Your table in `04` §6.) [Week 6 · 04 §6 ↗](../week6/04_convolution.ipynb)

</details>

**Quick check:** In a ResNet block, why does the skip path sometimes need a 1×1 convolution?

<details><summary>Answer</summary>

`out + skip` adds element by element, so both must have the same shape. When the block changes the number of channels or halves the size (stride 2), the 1×1 convolution (with that stride) converts the input to the new shape. [Week 6 · 05 §5 ↗](../week6/05_cnn_in_pytorch.ipynb)

</details>

**Code from memory: booklet 18.5.** You wrote in your log that "the code for max pooling was kind of hard to understand", then wrote a very good explanation of it into `04` (the 1-D version first, then the 4-part address). §A draws that explanation. Write the function closed book first.

**Exercise 1 · Booklet 18.5: max pooling with reshape.** `x` is one 2-D map `(h, w)` with `h` and `w` multiples of `size`. Return the `(h // size, w // size)` map whose entry `[I, J]` is the largest value in window `(I, J)`. One line: reshape to `(h // size, size, w // size, size)`, then take the max over axes 1 and 3.

**Example**: what goes in, and what should come out:
```python
x = np.array([[1, 3, 2, 0], [4, 2, 1, 1], [0, 1, 5, 2], [2, 0, 3, 4.0]])   # (4, 4) float
maxpool2d(x)    # → array([[4., 2.],
                #          [2., 5.]])
```

In [ ]:
def maxpool2d(x, size=2):
    pass  # your code

In [ ]:
check(1)

In [ ]:
check(1)

## A. Your Week 6, reviewed

You logged 11 hours over Nov 24–27 (`02` to `05`), and ticked `06` and `07` on Nov 28. Your notes this week were longer and better than in any week so far. Here's each one, checked.

| Where | What you wrote | Verdict |
|---|---|---|
| [02 §2 ↗](../week6/02_regularization.ipynb) | weight decay derived step by step: gradient $\nabla L + \lambda w$, then grouped into $w_{t+1} = (1 - \eta\lambda) w_t - \eta \nabla L$, "decay, then the usual step" | Correct, and it's the clearest form of the result. |
| [02 §3 ↗](../week6/02_regularization.ipynb) | inverted dropout: the $1/q$ mismatch between training and test, the fix, and "what it doesn't fix": the variance $a^2 (1 - q)/q$, and $\mathbb{E}[\sigma(\cdot)] \neq \sigma(\mathbb{E}[\cdot])$ | Correct, including the two limits, which most textbooks skip. Check of the variance: $\tilde a$ is $a/q$ with probability $q$, else 0, so $\mathbb{E}[\tilde a^2] = a^2/q$ and $\text{Var} = a^2/q - a^2 = a^2(1-q)/q$. |
| [03 §1 ↗](../week6/03_training_stability.ipynb) | the moving target: layer 5 tuned to inputs in $[-1, 1]$, then layer 4 shifts them to $[0, 4]$; the archery analogy; why the centre matters, not only the scale | Correct. This is exactly the motivation the batch-norm paper gives ("internal covariate shift"). |
| [03 §2 ↗](../week6/03_training_stability.ipynb) | why $\gamma$ and $\beta$: 200 parameters for 100 features; why `bias=False` before batch norm | Correct. |
| [03 §2 ↗](../week6/03_training_stability.ipynb) | batch norm at test time: Problems A and B, "Celsius vs Fahrenheit", running mean $\leftarrow 0.9 \times$ old $+ 0.1 \times$ batch | Correct. One detail for the exam: PyTorch's `momentum=0.1` is the weight on the NEW batch (the opposite of the optimizer's momentum 0.9), and the running variance uses the unbiased batch variance (divide by $n - 1$). |
| [04 §4 ↗](../week6/04_convolution.ipynb) | colour images as three stacked grids; PyTorch channels first `(3, 20, 20)`, NumPy/matplotlib channels last `(20, 20, 3)` | Correct, and the channels-first/last point is a common source of bugs. |
| [04 §5 ↗](../week6/04_convolution.ipynb) | why pooling: fewer numbers, and a 9 moved by one pixel gives the same pooled output (your drawing) | Correct, and the drawing shows it better than mine did. |
| [04 §7 ↗](../week6/04_convolution.ipynb) | the reshape trick for max pooling: the 1-D version, then the address `x4[I, a, J, b]` with $I$ = `r // 2`, $a$ = `r % 2` | Correct. It's drawn below so you can see it in one picture. |
| [05 §5 ↗](../week6/05_cnn_in_pytorch.ipynb) | the ResNet block as a diagram: main path and skip path, "must be the SAME shape to add" | Correct. |

🖼 **Picture.** Your explanation of the reshape trick, drawn. Every pixel gets a window address $(I, J)$ and a place $(a, b)$ inside that window; `reshape(2, 2, 2, 2)` just re-labels them, and `.max(axis=(1, 3))` collapses the within-window axes.

In [ ]:
fig_maxpool_reshape()

**Your code.** Every exercise passes; nothing to correct. Two things worth noticing: your `inverted_dropout` returns the mask too (`return (h * mask) / (1 - p), mask`), which is exactly what the backward pass needs; and you tested `multivariate_normal(...).pdf(X)` on a tiny input before writing the E-step. That habit (run the library call on two numbers first) is the fastest way to learn any package.

**Scores, again.** `06` and `07` are ticked, but their scores aren't in the log. Now that the exam is scored (§B), this matters: please add them (paper practice /100, quiz A /10, B /3, C /5) and list the misses **by topic**, so `05` and `06` this week can target them.

## B. After the meeting: aim for a high score

Prof. Yoon will take you into the lab; the exam is still on, but it measures how much you know rather than deciding whether you get in. So the goal changes from "enough to pass" to "as high as possible", and three things follow.

- **Coverage first.** The exam covers the 3rd–4th-year ML and DL courses. ML is done (Weeks 2–4, mock exam above 90). DL has two weeks left after this one: sequence models and attention (Transformer, ViT) in Week 8, then self-supervised learning and a full DL mock exam.
- **Hand calculations and code reading are where points are.** Every notebook this week ends with numbers to compute by hand, and `05` is a timed paper exam. Do it on paper.
- **Tell me the date** as soon as he sets it, and I'll fit the last weeks to it.

## C. This week's coding

Same rule as Week 6: every exercise is a booklet "In Python" block written from memory, or a hand-written piece of a new block at the same level (the core lines by hand, layers and optimizers from PyTorch, one check against the package).

| Notebook | Warm-up (booklet block) | New block |
|---|---|---|
| `00` | 18.5 max pooling |  |
| `01` transfer learning | 17.3 layer norm | freeze, replace the head, count what trains, two learning rates |
| `02` autoencoders | 6.6 PCA, reconstruction | the autoencoder class, per-image error, a denoising step, the transposed-convolution size |
| `03` the VAE | 9.4 the EM M-step | reparameterization, the KL term, the VAE loss |
| `04` anomaly detection | 8.5 DBSCAN's core points | Mahalanobis scores, a threshold, precision and recall, average precision |
| `05`, `06` | Weeks 3–6, spaced | this week's blocks |

The new blocks are written so each can become the "In Python" section of a booklet chapter (Transfer learning, Autoencoders, VAE, Anomaly detection) when you write your review notes.

## D. Pictures from now on

You wrote: "try to put more effort into the figures, compare your `fig_receptive_field()` in the ipynb and the figure that i had you draw for the review booklet". Side by side, the difference is clear:

| Mine (Week 6 · 04 §6) | Yours (booklet 18.6) |
|---|---|
| three flat 9×9 grids with a shaded square | the three layers stacked in 3-D, image at the bottom |
| shows the result ("5×5 is shaded") | shows the mechanism: dashed lines from one layer-2 value down to the 3×3 it reads, and from those down to the 5×5 |
| numbers only in the title | the counts on the picture itself: "1 value", "3×3 = 9 values", "5×5 = 25 pixels" |
| one colour | one colour per layer, kept consistent |

So from now on every figure follows your drawing's rules: **(1)** draw the mechanism (what feeds into what, with arrows or lines), not only the end result; **(2)** put the real numbers and shapes on the picture where they belong; **(3)** one colour per role, kept the same across a notebook (this week: blue = encoder or source, orange = decoder or what trains, green = the code, grey = frozen, red = gradients and anomalies); **(4)** a title that says what to look at. The max-pooling figure in §A is the first one drawn this way.

---
Solutions: `solutions/00_solutions.py`. Next: `01_transfer_learning.ipynb`.